# Import library

In [1]:
import os
import json
import time
import zipfile
import datetime
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import wandb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from utilis import Utility, Data, Visualization, Score, save_config_and_lr, load_config_and_lr
from chunk_handle import get_chunk_indices, iter_chunks, compute_global_image_stats, compute_global_label_stats, save_transform_and_scaler, load_transform_and_scaler

In [2]:
# chunk_dir= "./dataset/chunk_kappa_noise_new"
# indices=np.arange(10)
# means, stds=compute_global_image_stats(chunk_dir, indices)
# label_scaler=compute_global_label_stats(chunk_dir, indices)

In [3]:
# from torchvision import transforms
# transform = transforms.Compose([
#     transforms.ToTensor(),     
#     transforms.Normalize(mean=[means], std=[stds]),   
# ])
# print(f"Image stats (from train set): Mean={means}, Std={stds}")
# print(f"Label stats (from train set): Mean={label_scaler.mean_}, Std={np.sqrt(label_scaler.var_)}")

# save_transform_and_scaler(means, stds, label_scaler, transform_file='./side_module/transform_params.pkl', scaler_file='./side_module/label_scaler.pkl')

# If the transform and scaler is avaiable, ignore that

In [4]:
transform, label_scaler = load_transform_and_scaler(transform_file='./side_module/transform_params.pkl', scaler_file='./side_module/label_scaler.pkl')


Loaded transform with Mean=-0.00016638042870908976, Std=0.02047532983124256
Loaded label scaler with Mean=[0.29021683 0.81345297], Std=[0.1055216  0.06600116]


In [5]:
l_edge = np.logspace(2, 4, 11)
Nbins = len(l_edge)-1

In [6]:
def power_spectrum(x, pixsize, kedge):
    """
    Compute the azimuthally averaged 2D power spectrum of a real-valued 2D field.

    Parameters:
    -----------
    x : 2D numpy array
        Input real-space map (e.g., an image or simulated field).
        Must be a 2D array with shape (N_y, N_x).
    
    pixsize : float
        Physical size of each pixel in the map (e.g., arcmin, Mpc, etc.).
        Units should be consistent with the units used for `kedge`.
    
    kedge : 1D array-like
        Bin edges in wavenumber space (k), used to bin the power spectrum.
        Should be monotonically increasing and cover the k-range of interest.

    Returns:
    --------
    power_k : 1D numpy array
        The average wavenumber in each k bin (excluding the DC bin).
    
    power : 1D numpy array
        The binned, azimuthally averaged power spectrum corresponding to `power_k`.
        Normalized per unit area.
    """

    # Ensure the input array is 2D
    assert x.ndim == 2

    # Compute the 2D FFT of the input map and take its squared magnitude (power spectrum)
    xk = np.fft.rfft2(x)  # Real-to-complex FFT (along last axis)
    xk2 = (xk * xk.conj()).real  # Power spectrum: |FFT|^2

    # Get the shape of the input map
    Nmesh = x.shape

    # Compute the wavenumber grid (k-space)
    k = np.zeros((Nmesh[0], Nmesh[1]//2+1))
    # Square of the frequency in the first axis
    k += np.fft.fftfreq(Nmesh[0], d=pixsize).reshape(-1, 1) ** 2
    # Square of the frequency in the second axis (real FFT)
    k += np.fft.rfftfreq(Nmesh[1], d=pixsize).reshape(1, -1) ** 2
    # Convert from (1/length)^2 to angular frequency in radian units
    k = k ** 0.5 * 2 * np.pi

    # Bin each k value according to the bin edges provided in kedge
    index = np.searchsorted(kedge, k)

    # Bin the power values, number of modes, and wavenumbers
    power = np.bincount(index.flatten(), weights=xk2.flatten())
    Nmode = np.bincount(index.flatten())
    power_k = np.bincount(index.flatten(), weights=k.flatten())

    # Adjust for symmetry in the real FFT: include the mirrored part (excluding Nyquist frequency)
    if Nmesh[1] % 2 == 0:  # Even number of columns
        power += np.bincount(index[...,1:-1].flatten(), weights=xk2[...,1:-1].flatten())
        Nmode += np.bincount(index[...,1:-1].flatten())
        power_k += np.bincount(index[...,1:-1].flatten(), weights=k[...,1:-1].flatten())
    else:  # Odd number of columns
        power += np.bincount(index[...,1:].flatten(), weights=xk2[...,1:].flatten())
        Nmode += np.bincount(index[...,1:].flatten())
        power_k += np.bincount(index[...,1:].flatten(), weights=k[...,1:].flatten())

    # Exclude the first bin (typically corresponds to DC mode)
    power = power[1:len(kedge)]
    Nmode = Nmode[1:len(kedge)]
    power_k = power_k[1:len(kedge)]

    # Average the power and wavenumber in each bin, only where Nmode > 0
    select = Nmode > 0
    power[select] = power[select] / Nmode[select]
    power_k[select] = power_k[select] / Nmode[select]

    # Normalize the power spectrum by the map area
    power *= pixsize ** 2 / Nmesh[0] / Nmesh[1]

    # Return the binned k values and corresponding power spectrum
    return power_k, power

In [7]:
num_chunks=25
indices=np.arange(num_chunks)

In [8]:
pixelsize_arcmin = 2 # pixel size in arcmin
pixelsize_radian = pixelsize_arcmin / 60 / 180 * np.pi

In [9]:
# for chunk_idx in tqdm(indices, desc="Chunks", leave=False):
#             # Load single chunk (assumes it fits in memory)
#             noisy_chunk, label_chunk, _ = next(iter_chunks("./dataset/chunk_kappa_noise_new", indices=[chunk_idx]))
#             Ncosmo, Nsys = noisy_chunk.shape[0], noisy_chunk.shape[1]
#             Cl =  np.zeros((Ncosmo, Nsys, Nbins))

#             for i in range(Ncosmo):
#                 for j in range(Nsys):
#                     l, Cl[i,j] = power_spectrum(noisy_chunk[i,j].astype(np.float32
#                 ), pixelsize_radian, l_edge)

#             Utility.save_np(data_dir="./dataset/power_scpectrum", file_name=f"Spec_chunk_{chunk_idx}.npy",data=Cl)

In [10]:
# for chunk_idx in tqdm(indices, desc="Chunks", leave=False):
#     noisy_chunk, label_chunk, _ = next(iter_chunks("./dataset/chunk_kappa_noise_new", indices=[chunk_idx]))
#     Ncosmo, Nsys = noisy_chunk.shape[0], noisy_chunk.shape[1]
#     logCl = np.log10(Utility.load_np(data_dir="./dataset/power_scpectrum", file_name=f"Spec_chunk_{chunk_idx}.npy"))
#     mean_logCl = np.mean(logCl, 1)
#     Utility.save_np(data_dir="./dataset/meanlogPS", file_name=f"Spec_chunk_{chunk_idx}.npy",data=mean_logCl)
#     delta = (logCl - mean_logCl[:,None])
#     cov_logCl = [(delta[i].T @ delta[i] / (len(delta[i])-delta.shape[-1]-2))[None] for i in range(Ncosmo)]
#     cov_logCl = np.concatenate(cov_logCl, 0)
#     Utility.save_np(data_dir="./dataset/CovlogPS", file_name=f"Spec_chunk_{chunk_idx}.npy", data=cov_logCl)
#     print(f"shape of cov_logCl: {cov_logCl.shape}, shape of mean_logCl: {mean_logCl.shape}")

# Model architecture

In [11]:
from Models import Spectrum_CNN

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from vit_pytorch import ViT
from torch.utils.data import DataLoader, TensorDataset
import numpy as np


In [13]:

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), desc="Training")
    for (images, specs), targets in train_loader:
        images = images.to(device, dtype=torch.float32)
        specs = specs.to(device, dtype=torch.float32)
        targets = targets.to(device, dtype=torch.float32)
        optimizer.zero_grad()
        outputs = model((images, specs))
        # print(f"input shape: {inputs.shape}, target shape: {targets.shape}")
        # print(f"num of cosmo:{len(targets[:,0].unique())}")
        # print(f"num of cosmo_2:{len(targets[:,1].unique())}")

        # plt.hist(targets[:,0].cpu().numpy().flatten(), bins=500, alpha=0.5) # Giải nén tuple từ DataLoader
        # file_name = "histogram.png"  # Bạn có thể thay đổi tên file
        # save_path = os.path.join("./plt", file_name)
        # plt.savefig(save_path, dpi=300, bbox_inches='tight')  # Lưu với độ phân giải 300 DPI, căn chỉnh gọn gàng
        # # Đóng biểu đồ để tránh hiển thị trên màn hình (nếu không cần)
        # plt.close() 


        # inputs, targets = inputs.to(device), targets.to(device)
        
        # optimizer.zero_grad()
        # outputs = model(inputs)
        # print(outputs)
        # print("Outputs shape:", outputs.shape)  # In ra kích thước của outputs để kiểm tra
        # print("Targets shape:", targets.shape)  # In ra kích thước của targets để kiểm tra
        loss = criterion(outputs, targets)  # Gọi hàm mất mát
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate_epoch(model, dataloader, loss_fn, device):
    """Validates the model on the validation/test set."""
    model.eval()
    total_loss = 0
    pbar = tqdm(dataloader, total=len(dataloader), desc="Validating")
    with torch.no_grad():
        for (images, specs), targets in dataloader:
            images = images.to(device, dtype=torch.float32)
            specs = specs.to(device, dtype=torch.float32)
            targets = targets.to(device, dtype=torch.float32)
            outputs = model((images, specs))
            total_loss += loss_fn(outputs, targets).item()
            
    return total_loss / len(dataloader)


In [20]:
from train import Trainer

In [ ]:

class Config:
    IMG_HEIGHT = 1424
    IMG_WIDTH = 176
    
    # Parameters to predict (Omega_m, S_8, sigma_Omega_m, sigma_S_8)
    NUM_TARGETS = 4

    # Training hyperparameters
    BATCH_SIZE = 101
    EPOCHS = 15
    LEARNING_RATE = 1e-6
    WEIGHT_DECAY = 1e-4   # L2 regularization to prevent overfitting
    IMG_RESIZE = 256  # Resize images to 256x256 for ViT
    DEVICE = "mps" if torch.has_mps else "cpu"
    MODEL_SAVE_PATH = None  # Will be set dynamically with timestamp


config = Config()
# model = CustomViT()




model=Spectrum_CNN(config.IMG_HEIGHT, config.IMG_WIDTH, 2)
model_name="Spectrum_CNN"


# model = CustomViT()
# model_name="Custom_ViT"

# pretrain =False
pretrain=True
previous_path = "/Users/viethuy/Working_space/Neurips/Neurips2025_Weak_lensing/side_module_ViT/model_20250921_015923_on"  # Set to the timestamp of the previous run if pretrain=True

if pretrain:
    if previous_path is None:
        raise ValueError("Please specify previous_timestamp for pretraining (e.g., '20250918_144500').")
    # config_file = f'{previous_path}/training_config.pkl'
    # config, last_lr = load_config_and_lr(config_file=config_file)
    last_lr=1e-4
    model.load_state_dict(torch.load(f"{previous_path}/best_model.pth", weights_only=True))
    optimizer = optim.Adam(model.parameters(), lr=last_lr, weight_decay=config.WEIGHT_DECAY)
else:
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

# criterion = KL_div_posterior_loss  # Replace with your criterion
criterion = nn.MSELoss()  # Replace with your criterion

run_name = f"{model_name}_run_{time.strftime('%Y_%m_%d_%H_%M_%S')}"

# Initialize W&B run
wandb.init(
    entity="huy_ben",
    project="cosmology_training",
          # project name (string, not f-string with stray bracket)
    name=run_name,   
    config={
        "IMG_HEIGHT": config.IMG_HEIGHT,
        "IMG_WIDTH": config.IMG_WIDTH,
        "NUM_TARGETS": config.NUM_TARGETS,
        "BATCH_SIZE": config.BATCH_SIZE,
        "EPOCHS": config.EPOCHS,
        "LEARNING_RATE": config.LEARNING_RATE,
        "WEIGHT_DECAY": config.WEIGHT_DECAY,
        "DEVICE": config.DEVICE,
        "IMG_RESIZE": config.IMG_RESIZE,
        "split_ratio": 0.8,
        "epochs_per_chunk": 1
})



# Continue training
history = Trainer.Train_chunk(
    model=model,
    config=config,
    optimizer=optimizer,
    criterion=criterion,
    chunk_dir='./dataset/chunk_kappa_noise_new',
    val_size=0.2,
    transform=transform,
    label_scaler=label_scaler,
    verbose=False
)

/var/folders/y7/lp_fcj6s2bn7220l1k8jhbh40000gn/T/ipykernel_51110/3349588138.py:14: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  DEVICE = "mps" if torch.has_mps else "cpu"


Epochs:   0%|          | 0/15 [00:02<?, ?it/s]


TypeError: CosmologyDataset.__init__() got an unexpected keyword argument 'specs'

In [ ]:
# for chunk_idx in tqdm(indices, desc="Chunks", leave=False):
#             # Load single chunk (assumes it fits in memory)
#             noisy_chunk, label_chunk, _ = next(iter_chunks("./dataset/chunk_kappa_noise_new", indices=[chunk_idx]))
#             Ncosmo, Nsys = noisy_chunk.shape[0], noisy_chunk.shape[1]
#             Cl =  np.zeros((Ncosmo, Nsys, Nbins))

#             for i in range(Ncosmo):
#                 for j in range(Nsys):
#                     l, Cl[i,j] = power_spectrum(noisy_chunk[i,j].astype(np.float32
#                 ), pixelsize_radian, l_edge)

#             Utility.save_np(data_dir="./dataset/power_scpectrum", file_name=f"Spec_chunk_{chunk_idx}.npy",data=Cl)

In [ ]:
import os
import numpy as np
import pickle
from torch.utils.data import DataLoader

# Giả sử các biến như transform, config, CosmologyDataset đã được định nghĩa
total_val_datasets = history['total_val_datasets']
train_losses = history['train_losses']
total_samples = history['total_samples']
total_epochs = history['total_epochs']

save_dir = f"./dataset/val_set/"
os.makedirs(save_dir, exist_ok=True)

# Đường dẫn file lưu trữ
val_data_path = os.path.join(save_dir, "all_val_data.npy")
val_labels_path = os.path.join(save_dir, "all_val_labels.npy")
val_specs_path = os.path.join(save_dir, "all_val_specs.npy")
metadata_path = os.path.join(save_dir, "validation_metadata.pkl")

if total_val_datasets:
    # Kiểm tra nếu file đã tồn tại, tải lại nếu có
    if os.path.exists(val_data_path) and os.path.exists(val_labels_path) and os.path.exists(val_specs_path):
        all_val_data = np.load(val_data_path)
        all_val_labels = np.load(val_labels_path)
        Cl = np.load(val_specs_path)
        print(f"Loaded validation data from {save_dir}")
    else:
        all_val_data = np.concatenate([ds.data for ds in total_val_datasets], axis=0)
        all_val_labels = np.concatenate([ds.labels for ds in total_val_datasets], axis=0)
        Nval = all_val_data.shape[0]
        Cl = np.zeros((Nval, Nbins))

        for i in range(Nval):
            l, Cl[i] = power_spectrum(all_val_data[i].astype(np.float32), pixelsize_radian, l_edge)
        
        # Lưu dữ liệu vào file
        np.save(val_data_path, all_val_data)
        np.save(val_labels_path, all_val_labels)
        np.save(val_specs_path, Cl)
        print(f"Saved validation data to {save_dir}")

    # Lưu metadata (config, batch_size, v.v.) để tái sử dụng
    metadata = {
        "batch_size": config.BATCH_SIZE,
        "transform": transform,  # Lưu transform nếu có thể serialize
        "shape": all_val_data.shape
    }
    with open(metadata_path, 'wb') as f:
        pickle.dump(metadata, f)

    # Tạo dataset và dataloader
    concatenated_val_dataset = CosmologyDataset(
        data=all_val_data,
        labels=all_val_labels,
        specs=Cl,
        transform=transform  # Use transform from first dataset
    )
    val_loader = DataLoader(
        concatenated_val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False
    )
else:
    concatenated_val_loader = None
    print("No validation datasets to concatenate.")

In [ ]:
# Nval= all_val_data.shape[0]
# Cl =  np.zeros((Nval, Nbins))
# for i in range(Nval):
#     l, Cl[i] = power_spectrum(all_val_data[i].astype(np.float32), pixelsize_radian, l_edge)
        
#         # Lưu dữ liệu vào file
# np.save(val_data_path, all_val_data)
# np.save(val_labels_path, all_val_labels)
# np.save(val_specs_path, Cl)
# print(f"Saved validation data to {save_dir}")

#     # Lưu metadata (config, batch_size, v.v.) để tái sử dụng
# metadata = {
#         "batch_size": config.BATCH_SIZE,
#         "transform": transform,  # Lưu transform nếu có thể serialize
#         "shape": all_val_data.shape
#     }
# with open(metadata_path, 'wb') as f:
#         pickle.dump(metadata, f)

#     # Tạo dataset và dataloader
# concatenated_val_dataset = CosmologyDataset(
#         data=all_val_data,
#         labels=all_val_labels,
#         specs=Cl,
#         transform=transform  # Use transform from first dataset
#     )
# val_loader = DataLoader(
#         concatenated_val_dataset,
#         batch_size=config.BATCH_SIZE,
#         shuffle=False
#     )

In [ ]:
# total_val_datasets = history['total_val_datasets']
# train_losses = history['train_losses']
# total_samples = history['total_samples']
# total_epochs = history['total_epochs']
# if total_val_datasets:
#     all_val_data = np.concatenate([ds.data for ds in total_val_datasets], axis=0)
#     all_val_labels = np.concatenate([ds.labels for ds in total_val_datasets], axis=0)
#     # noisy_chunk, label_chunk, _ = next(iter_chunks("./dataset/chunk_kappa_noise_new", indices=[chunk_idx]))
#     Ncosmo, Nsys = all_val_data.shape[0], all_val_data.shape[1]
#     Cl =  np.zeros((Ncosmo, Nsys, Nbins))

#     for i in range(Ncosmo):
#         for j in range(Nsys):
#             l, Cl[i,j] = power_spectrum(all_val_data[i,j].astype(np.float32), pixelsize_radian, l_edge)
#     concatenated_val_dataset = CosmologyDataset(
#         data=all_val_data,
#         labels=all_val_labels,
#         specs=Cl,
#         transform=transform # Use transform from first dataset
#     )
#     val_loader = DataLoader(
#         concatenated_val_dataset,
#         batch_size=config.BATCH_SIZE,
#         shuffle=False
#     )
# else:
#     concatenated_val_loader = None
#     print("No validation datasets to concatenate.")

In [ ]:
# for chunk_idx in tqdm(indices, desc="Chunks", leave=False):
#             # Load single chunk (assumes it fits in memory)
#             noisy_chunk, label_chunk, _ = next(iter_chunks("./dataset/chunk_kappa_noise_new", indices=[chunk_idx]))
#             Ncosmo, Nsys = noisy_chunk.shape[0], noisy_chunk.shape[1]
#             Cl =  np.zeros((Ncosmo, Nsys, Nbins))

#             for i in range(Ncosmo):
#                 for j in range(Nsys):
#                     l, Cl[i,j] = power_spectrum(noisy_chunk[i,j].astype(np.float32
#                 ), pixelsize_radian, l_edge)

#             Utility.save_np(data_dir="./dataset/power_scpectrum", file_name=f"Spec_chunk_{chunk_idx}.npy",data=Cl)

In [ ]:
model.eval()
device=config.DEVICE
y_pred_list = []   
pbar = tqdm(val_loader, total=len(val_loader), desc="Validating")
with torch.no_grad():
    for (images, specs), targets in val_loader:
        images = images.to(device, dtype=torch.float32)
        specs = specs.to(device, dtype=torch.float32)
        targets = targets.to(device, dtype=torch.float32)
        y_pred = model((images, specs))
        y_pred = label_scaler.inverse_transform(y_pred.cpu().numpy())
        y_pred_list.append(y_pred) 

mean_val = np.concatenate(y_pred_list, axis=0)

In [ ]:
# Comparison of the means & standard deviations of the posterior distributions and the validation labels
all_val_labels_inv = label_scaler.inverse_transform(all_val_labels)
plt.errorbar(all_val_labels_inv[:,0], mean_val[:,0], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(all_val_labels_inv[:,0]), sorted(all_val_labels_inv[:,0]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(all_val_labels_inv[:,0]), np.max(all_val_labels_inv[:,0]))
plt.ylim(0, 0.7)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$\Omega_m$')
plt.show()

plt.errorbar(all_val_labels_inv[:,1], mean_val[:,1], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(all_val_labels_inv[:,1]), sorted(all_val_labels_inv[:,1]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(all_val_labels_inv[:,1]), np.max(all_val_labels_inv[:,1]))
plt.ylim(0.65, 1)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$S_8$')
plt.show()

In [ ]:
# Initialize Data class object
data_obj = Data(data_dir="./dataset", USE_PUBLIC_DATASET=True)

# Load train data
data_obj.load_train_data()

# Load test data
data_obj.load_test_data()

In [ ]:
Ncosmo = data_obj.Ncosmo
Nsys = data_obj.Nsys

print(f'There are {Ncosmo} cosmological models, each has {Nsys} realizations of nuisance parameters in the training data.')

In [ ]:
# cosmology: (Ncosmo, 2)
# mean_val: (Nval, 2)
cosmology = data_obj.label[:,0,:2]   
dists = np.sum((all_val_labels_inv[:, None, :] - cosmology[None, :, :])**2, axis=2)  # (Nval, Ncosmo)
nearest_idx = np.argmin(dists, axis=1)  # for each validation sample, index of closest cosmology

# build index lists grouped by cosmology
index_lists = [[] for _ in range(cosmology.shape[0])]
for val_idx, cosmo_i in enumerate(nearest_idx):
    index_lists[cosmo_i].append(val_idx)

val_cosmology_idx = [np.array(lst, dtype=np.int32) for lst in index_lists]

In [ ]:
# # There are Ncosmo distinct cosmologies in the labels.
# # Here we create a list that groups the indices of the validation instances with the same cosmological parameters

# cosmology = data_obj.label[:,0,:2]   # shape = (Ncosmo, 2)

# row_to_i = {tuple(cosmology[i]): i for i in range(Ncosmo)}
# index_lists = [[] for _ in range(cosmology.shape[0])]

# # Loop over each row in 'y_val' with shape = (Nval, 2)
# for idx in range(len(all_val_labels_inv)):
#     row_tuple = tuple(all_val_labels_inv[idx])
#     i = row_to_i[row_tuple]
#     index_lists[i].append(idx)

# # val_cosmology_idx[i] = the indices idx of the validation examples with labels = cosmology[i]
# val_cosmology_idx = [np.array(lst) for lst in index_lists]  

In [ ]:
# The summary statistics of all realizations for all cosmologies in the validation set
d_vector = []  
n_d = 2   # Number of summary statistics for each map
for i in range(Ncosmo):
    d_i =  np.zeros((len(val_cosmology_idx[i]), n_d))  
    for j, idx in enumerate(val_cosmology_idx[i]):
        d_i[j] = mean_val[idx]

    d_vector.append(d_i)

In [ ]:
# mean summary statistics (average over all realizations)
mean_d_vector = []
for i in range(Ncosmo):
    mean_d_vector.append(np.mean(d_vector[i], 0))
mean_d_vector = np.array(mean_d_vector)   

# covariance matrix
delta = []
for i in range(Ncosmo):
    delta.append((d_vector[i] - mean_d_vector[i].reshape(1, n_d))) 

cov_d_vector = [(delta[i].T @ delta[i] / (len(delta[i])-n_d-2))[None] for i in range(Ncosmo)]     
cov_d_vector = np.concatenate(cov_d_vector, 0) 

In [ ]:
from scipy.interpolate import LinearNDInterpolator
mean_d_vector_interp = LinearNDInterpolator(cosmology, mean_d_vector, fill_value=np.nan)
cov_d_vector_interp = LinearNDInterpolator(cosmology, cov_d_vector, fill_value=np.nan)

In [ ]:
logprior_interp = LinearNDInterpolator(cosmology, np.zeros((Ncosmo, 1)), fill_value=-np.inf)

# Note that the training data are not uniformly sampled, which introduces a prior distribution. Here we ignore that prior for simplicity.
# Also note that this prior would introduce bias for cosmologies at the boundary of the prior
def log_prior(x):
    logprior = logprior_interp(x).flatten()  # shape = (Ntest, ) 
    return logprior

# Gaussian likelihood with interpolated mean and covariance matrix
def loglike(x, d):
    mean = mean_d_vector_interp(x) 
    cov = cov_d_vector_interp(x)   
    delta = d - mean               
    
    inv_cov = np.linalg.inv(cov)
    cov_det = np.linalg.slogdet(cov)[1]
    
    return -0.5 * cov_det - 0.5 * np.einsum("ni,nij,nj->n", delta, inv_cov, delta)

def logp_posterior(x, d):
    logp = log_prior(x)
    select = np.isfinite(logp)
    if np.sum(select) > 0:
        logp[select] = logp[select] + loglike(x[select], d[select])
    return logp

In [ ]:
Nval=all_val_labels_inv.shape[0]
print(Nval)

In [ ]:
mean_val.shape

In [ ]:
# MCMC sampling to explore the posterior distribution

Nstep = 10000  # Number of MCMC steps (iterations)
sigma = 0.06   # Proposal standard deviation; should be tuned per method or parameter scale

# Randomly select initial points from the `cosmology` array for each test case
# Assumes `cosmology` has shape (Ncosmo, ndim) and `Ntest` is the number of independent chains/samples
current = cosmology[np.random.choice(Ncosmo, size=Nval)]

# Compute log-posterior at the initial points
curr_logprob = logp_posterior(current, mean_val)

# List to store sampled states (for all chains)
states = []

# Track total acceptance probabilities to compute acceptance rates
total_acc = np.zeros(len(current))

t = time.time()  # Track time for performance reporting

# MCMC loop
for i in range(Nstep):

    # Generate proposals by adding Gaussian noise to current state
    proposal = current + np.random.randn(*current.shape) * sigma    

    # Compute log-posterior at the proposed points
    proposal_logprob = logp_posterior(proposal, mean_val)

    # Compute log acceptance ratio (Metropolis-Hastings)
    acc_logprob = proposal_logprob - curr_logprob
    acc_logprob[acc_logprob > 0] = 0  # Cap at 0 to avoid exp overflow (acceptance prob ≤ 1)

    # Convert to acceptance probabilities
    acc_prob = np.exp(acc_logprob)

    # Decide whether to accept each proposal
    acc = np.random.uniform(size=len(current)) < acc_prob

    # Track acceptance probabilities (not binary outcomes)
    total_acc += acc_prob

    # Update states and log-probs where proposals are accepted
    current[acc] = proposal[acc]
    curr_logprob[acc] = proposal_logprob[acc]

    # Save a copy of the current state
    states.append(np.copy(current)[None])

    # Periodically print progress and acceptance rates
    if i % (0.1*Nstep) == 0.1*Nstep-1:
        print(
            'step:', len(states),
            'Time:', time.time() - t,
            'Min acceptance rate:', np.min(total_acc / (i + 1)),
            'Mean acceptance rate:', np.mean(total_acc / (i + 1))
        )
        t = time.time()  # Reset timer for next print interval

In [ ]:
y_pred_val=mean_val

In [ ]:
# remove burn-in
states = np.concatenate(states[int(0.2*Nstep):], 0)

# mean and std of samples
mean_val = np.mean(states, 0)
errorbar_val = np.std(states, 0)

In [ ]:
mean_val

In [ ]:
y_val=all_val_labels_inv

In [ ]:
# Comparison of the means & standard deviations of the posterior distributions and the validation labels

plt.errorbar(y_val[:,0], mean_val[:,0], yerr=errorbar_val[:,0], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(y_val[:,0]), sorted(y_val[:,0]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(y_val[:,0]), np.max(y_val[:,0]))
plt.ylim(0, 0.7)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$\Omega_m$')
plt.show()

plt.errorbar(y_val[:,1], mean_val[:,1], yerr=errorbar_val[:,1], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(y_val[:,1]), sorted(y_val[:,1]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(y_val[:,1]), np.max(y_val[:,1]))
plt.ylim(0.65, 1)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$S_8$')
plt.show()

In [ ]:
validation_score = Score._score_phase1(
    true_cosmo=y_val,
    infer_cosmo=mean_val,
    errorbar=errorbar_val
)
print('averaged score:', np.mean(validation_score))
print('averaged error bar:', np.mean(errorbar_val, 0))

In [ ]:
test_Cl = np.zeros((data_obj.Ntest, Nbins))

for i in range(data_obj.Ntest):
    l, test_Cl[i] = power_spectrum(data_obj.kappa_test[i].astype(np.float64), data_obj.pixelsize_radian, l_edge)

test_logCl = np.log10(test_Cl)

In [ ]:
test_dataset = CosmologyDataset(
    data=data_obj.kappa_test, 
    specs=test_Cl,
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

In [ ]:
model.eval()
y_pred_list = []   
pbar = tqdm(test_loader, total=len(test_loader), desc="Inference on the test set")
with torch.no_grad():
    for images, specs in test_loader:
        images = images.to(device, dtype=torch.float32)
        specs = specs.to(device, dtype=torch.float32)
        y_pred = model((images, specs))      
        y_pred = label_scaler.inverse_transform(y_pred.cpu().numpy())
        y_pred_list.append(y_pred) 

y_pred_test = np.concatenate(y_pred_list, axis=0)

In [ ]:
y_pred_test

In [ ]:
# MCMC sampling to explore the posterior distribution

Nstep = 10000  # Number of MCMC steps (iterations)
sigma = 0.06   # Proposal standard deviation; should be tuned per method or parameter scale

# Randomly select initial points from the `cosmology` array for each test case
# Assumes `cosmology` has shape (Ncosmo, ndim) and `Ntest` is the number of independent chains/samples
current = cosmology[np.random.choice(Ncosmo, size=data_obj.Ntest)]

# Compute log-posterior at the initial points
curr_logprob = logp_posterior(current, y_pred_test)

# List to store sampled states (for all chains)
states = []

# Track total acceptance probabilities to compute acceptance rates
total_acc = np.zeros(len(current))

t = time.time()  # Track time for performance reporting

# MCMC loop
for i in range(Nstep):

    # Generate proposals by adding Gaussian noise to current state
    proposal = current + np.random.randn(*current.shape) * sigma    

    # Compute log-posterior at the proposed points
    proposal_logprob = logp_posterior(proposal, y_pred_test)

    # Compute log acceptance ratio (Metropolis-Hastings)
    acc_logprob = proposal_logprob - curr_logprob
    acc_logprob[acc_logprob > 0] = 0  # Cap at 0 to avoid exp overflow (acceptance prob ≤ 1)

    # Convert to acceptance probabilities
    acc_prob = np.exp(acc_logprob)

    # Decide whether to accept each proposal
    acc = np.random.uniform(size=len(current)) < acc_prob

    # Track acceptance probabilities (not binary outcomes)
    total_acc += acc_prob

    # Update states and log-probs where proposals are accepted
    current[acc] = proposal[acc]
    curr_logprob[acc] = proposal_logprob[acc]

    # Save a copy of the current state
    states.append(np.copy(current)[None])

    # Periodically print progress and acceptance rates
    if i % (0.1*Nstep) == 0.1*Nstep-1:
        print(
            'step:', len(states),
            'Time:', time.time() - t,
            'Min acceptance rate:', np.min(total_acc / (i + 1)),
            'Mean acceptance rate:', np.mean(total_acc / (i + 1))
        )
        t = time.time()  # Reset timer for next print interval

In [ ]:
# remove burn-in
states = np.concatenate(states[int(0.2*Nstep):], 0)

# mean and std of samples
mean = np.mean(states, 0)
errorbar = np.std(states, 0)

In [ ]:
states

In [ ]:
mean

In [ ]:
data = {"means": mean.tolist(), "errorbars": errorbar.tolist()}
the_date = datetime.datetime.now().strftime("%y-%m-%d-%H-%M")
zip_file_name = 'Submission_' + the_date + '.zip'
zip_file = Utility.save_json_zip(
    submission_dir="submissions",
    json_file_name="result.json",
    zip_file_name=zip_file_name,
    data=data
)
print(f"Submission ZIP saved at: {zip_file}")